In [ ]:
!pip install langchain-text-splitters langchain-community langchain-huggingface langchain-core transformers streamlit faiss-cpu
import streamlit as st
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, pipeline
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# --- Page Configuration ---
st.set_page_config(page_title="Smart Jeevan Shala AI Agent", page_icon="🤖")
st.title("🤖 Smart Jeevan Shala: RAG AI Assistant")
st.markdown("Retrieval-Augmented Generation system for Educational Content.")

# --- Initialize RAG System ---
@st.cache_resource
def initialize_rag():
    # 1. Create Knowledge Base
    content = """
    Smart Jeevan Shala focuses on the holistic development of students.
    One of the core modules is Financial Literacy for teenagers.
    In the Financial Literacy module, students learn about saving money, basic banking, and the power of compounding.
    A good budget follows the 50-30-20 rule: 50% for needs, 30% for wants, and 20% for savings.
    Emotional Intelligence helps students manage stress and make better financial decisions.
    """
    with open("kb.txt", "w") as f:
        f.write(content)

    # 2. Load and Split Documents
    loader = TextLoader("kb.txt")
    docs = loader.load()
    splitter = RecursiveCharacterTextSplitter(chunk_size=150, chunk_overlap=20)
    chunks = splitter.split_documents(docs)

    # 3. Create Vector Store
    embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
    vector_store = FAISS.from_documents(chunks, embeddings)
    retriever = vector_store.as_retriever(search_kwargs={"k": 2})

    # 4. Load LLM
    model_id = "google/flan-t5-large"
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_id)
    pipe = pipeline("text-generation", model=model, tokenizer=tokenizer, max_new_tokens=100)
    llm = HuggingFacePipeline(pipeline=pipe)

    # 5. Build RAG Chain
    template = """Use the following pieces of context to answer the question.
Context: {context}
Question: {question}
Answer:"""
    prompt = PromptTemplate.from_template(template)

    def format_docs(docs):
        return "\n\n".join(doc.page_content for doc in docs)

    chain = (
        {"context": retriever | format_docs, "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )
    return chain

# --- Load System ---
with st.spinner("Initializing AI Agent... Please wait."):
    rag_chain = initialize_rag()

# --- Chat Interface ---
if "messages" not in st.session_state:
    st.session_state.messages = []

for message in st.session_state.messages:
    with st.chat_message(message["role"]):
        st.markdown(message["content"])

if user_input := st.chat_input("Ask about Smart Jeevan Shala curriculum..."):
    st.session_state.messages.append({"role": "user", "content": user_input})
    with st.chat_message("user"):
        st.markdown(user_input)

    with st.chat_message("assistant"):
        with st.spinner("Searching Knowledge Base..."):
            response = rag_chain.invoke(user_input)
            clean_response = response.split("Answer:")[-1].strip()
            st.markdown(clean_response)
            st.session_state.messages.append({"role": "assistant", "content": clean_response})

# --- Sidebar ---
st.sidebar.title("System Info")
st.sidebar.info("This AI uses RAG to prevent hallucinations by only answering from the provided dataset.")

2026-05-12 18:17:11.334 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-12 18:17:11.337 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-12 18:17:11.340 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-12 18:17:11.341 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-12 18:17:11.342 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-12 18:17:11.343 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-12 18:17:11.344 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-12 18:17:11.347 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM

DeltaGenerator(_root_container=1, _parent=DeltaGenerator())

In [16]:
model_id = "google/flan-t5-base"  # use base, faster on Streamlit Cloud
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSeq2SeqLM.from_pretrained(model_id)

from transformers import pipeline # Reverting to import pipeline factory function
pipe = pipeline(
    "question-answering", # Using 'question-answering' as a workaround for T5 models
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=100
)
llm = HuggingFacePipeline(pipeline=pipe)

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The model 'T5ForConditionalGeneration' is not supported for question-answering. Supported models are ['PeftModelForQuestionAnswering', 'AlbertForQuestionAnswering', 'ArceeForQuestionAnswering', 'BartForQuestionAnswering', 'BertForQuestionAnswering', 'BigBirdForQuestionAnswering', 'BigBirdPegasusForQuestionAnswering', 'BloomForQuestionAnswering', 'CamembertForQuestionAnswering', 'CanineForQuestionAnswering', 'ConvBertForQuestionAnswering', 'Data2VecTextForQuestionAnswering', 'DebertaForQuestionAnswering', 'DebertaV2ForQuestionAnswering', 'DiffLlamaForQuestionAnswering', 'DistilBertForQuestionAnswering', 'ElectraForQuestionAnswering', 'ErnieForQuestionAnswering', 'Exaone4ForQuestionAnswering', 'FalconForQuestionAnswering', 'FlaubertForQues